# DataQualityAgent: детектив → хирург → аргумент

Notebook вызывает production-методы `detect_issues`, `evaluate_strategies` и `compare`. Исходный DataFrame не мутируется; source labels используются только для диагностики распределения.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'config.yaml').exists() and (candidate / 'agents').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from agents.common import load_frame
from agents.data_quality_agent import DataQualityAgent

quality = DataQualityAgent(ROOT / 'config.yaml')

In [ ]:
candidates = [
    ROOT / 'data/raw/reviews_raw.parquet',
    ROOT / 'data/raw/reviews_raw.csv',
    ROOT / 'data/fixtures/reviews_fixture.csv',
]
raw_path = next((path for path in candidates if path.exists()), None)
if raw_path is None:
    raw = None
    print('Нет входного датасета. Сначала запустите collection stage.')
else:
    raw = load_frame(raw_path)
    print(f'Input: {raw_path} · rows={len(raw)}')

## Часть 1 — детектив

Отдельно считаются null, пустые строки, полные дубликаты, IQR/z-score выбросы длины и отношение minority/majority. Это важно для текста: экстремально длинный отзыв может быть настоящим, поэтому сам факт IQR-флага ещё не означает, что строку следует удалить.

In [ ]:
if raw is not None:
    issues = quality.detect_issues(raw)
    headline = {
        'rows': issues['row_count'],
        'missing_cells': issues['missing_total'],
        'empty_text': issues['empty_text'],
        'duplicates': issues['duplicates'],
        'iqr_outliers': issues['outliers']['iqr']['count'],
        'zscore_outliers': issues['outliers']['zscore']['count'],
        'imbalance': issues['imbalance'],
    }
    display(headline)
else:
    print('Диагностика пропущена: входной DataFrame отсутствует.')

## Часть 2 — хирург: две стратегии на одном input

`conservative` удаляет строки без обязательных полей и полные дубликаты, а длинные тексты обрезает по верхней IQR-границе. `strict` вместо обрезки удаляет IQR-выбросы. Сравнение выполняется на одной неизменной исходной таблице.

In [ ]:
if raw is not None:
    strategy_results = quality.evaluate_strategies(raw, ['conservative', 'strict'])
    for name, payload in strategy_results.items():
        print(f'\nStrategy: {name} · rows={len(payload["dataframe"])}')
        display(payload['comparison'])
        display(payload['actions'])
else:
    strategy_results = {}
    print('Сравнение пропущено: входной DataFrame отсутствует.')

## Часть 3 — аргумент выбора

Для этой ML-задачи базовый выбор — **conservative**. Amazon и Steam закономерно отличаются длиной и стилем; удаление всех статистически длинных отзывов может непропорционально убрать один домен и полезные смешанные случаи. Обрезка ограничивает влияние экстремальной длины на вычисления, но сохраняет строку, источник и метку.

Стратегия не считается лучшей автоматически: после фактического запуска нужно проверить таблицу до/после, class/source distributions и примеры обрезанных текстов. `strict` оправдан только если ручной аудит покажет, что выбросы — мусор, HTML/логи или некорректный сбор. Ни одна стратегия не должна переписывать target labels.

In [ ]:
report_path = ROOT / 'reports/quality/quality_report.md'
if report_path.exists():
    display(Markdown(report_path.read_text(encoding='utf-8')))
else:
    print('Сохранённого quality report пока нет; его создаёт основной pipeline runner.')